In [ ]:

import numpy as np
import pandas as pd
import os
import rioxarray as rxr
from osgeo import gdal
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
import geopandas as gpd
import rasterio
from shapely.geometry import box
from shapely.geometry import Point
from rasterio.sample import sample_gen
import xgboost as xgb

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real

In [ ]:


def save_img(output_filename, reference_filename, array):
    reference = gdal.Open(reference_filename)
    
    # Get geotransform and projection from reference raster
    geo_transform = reference.GetGeoTransform()
    projection = reference.GetProjection()

    # Get array dimensions
    rows, cols = array.shape

    # Create a new TIFF dataset
    driver = gdal.GetDriverByName('GTiff')
    out_data = driver.Create(output_filename, cols, rows, 1, gdal.GDT_Float32)

    # Set geotransform and projection
    out_data.SetGeoTransform(geo_transform)
    out_data.SetProjection(projection)

    # Write array to raster
    out_data.GetRasterBand(1).WriteArray(array)

    # Save and close file
    out_data.FlushCache()
    out_data = None
    

def load_refer_icesat_point(FB_raster, icesat_csv_path, output_point, output_raw_point):
    # Load ICESat CSV as GeoDataFrame
    df = pd.read_csv(icesat_csv_path)
    geometry = [Point(xy) for xy in zip(df['mean_lon'], df['mean_lat'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    
    # Sample raster values at point locations
    with rasterio.open(FB_raster) as src:
        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)
        coords = [(point.x, point.y) for point in gdf.geometry]
        values = list(src.sample(coords))
        
        for band in range(src.count):
            gdf[f"b{band+1}_FB6"] = [v[band] for v in values]
    
    # Filter points based on ICESat height and raster band 7
    icesat_mask = (gdf['OH'] < 5) & (gdf['b7_FB6'] > 0)
    data = gdf[icesat_mask]
    
    # Save raw points and filtered points
    gdf.to_file(output_raw_point)
    data.to_file(output_point)
    
    return data


def optimize_random_forest(x_train, y_train, n_iter=30, cv=3):
    # Initialize Random Forest Regressor
    classifier = RandomForestRegressor(random_state=42)

    # Define hyperparameter search space
    search_space = {
        'n_estimators': Integer(50, 500),
        'max_depth': Integer(3, 20),
        'min_samples_split': Integer(2, 20),
        'min_samples_leaf': Integer(1, 20),
        'max_features': Real(0.1, 1.0)
    }

    # Perform Bayesian optimization
    opt = BayesSearchCV(
        estimator=classifier,
        search_spaces=search_space,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )
    opt.fit(x_train, y_train)

    best_params = opt.best_params_
    print('Best hyperparameters:', best_params)
    print('Best score (negative MSE):', opt.best_score_)

    # Train final model with optimized parameters
    final_model = RandomForestRegressor(**best_params, random_state=42)

    return best_params, final_model
    

In [ ]:
import os
import numpy as np
import rioxarray as rxr
import pandas as pd

# --------------------------- ICESat-2 data used to train RF model ---------------------------
region = 'Region4'

# List of locations to process
location_list = ['Morecambe Bay']

# List of years to process
year_list = [2019, 2020, 2021, 2022, 2023, 2024]

method = 'RF'
band = 4

for location in location_list:
    
    # Load Land Use/Land Cover (LULC) raster
    LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    LULC = rxr.open_rasterio(LULC_path, masked=True).squeeze()
    
    target_band, target_id = ['b1_FB6', 'b2_FB6', 'b3_FB6', 'b4_FB6'], [0, 1, 2, 3]

    # Load ICESat points for all years
    ICESat_points_list = []
    for year in year_list:
        FB_raster_path_ = f"G:/ICESat-2/{region}_map/{location}_map/A{year}/{location}_{year}_FB6_new9.tif"
        ICESAT_csv_path_ = f'G:/ICESat-2/{region}/{region}_Ribble/A_Ribble_30m_{year}_new10.csv'
        output_raw_point = f'G:/ICESat-2/{region}_map/{location}_map/A{year}/A_{location}_30m_{year}_raw_new10.shp'
        output_point = f'G:/ICESat-2/{region}_map/{location}_map/A{year}/A_{location}_30m_{year}_new10.shp'
        
        data = load_refer_icesat_point(FB_raster_path_, ICESAT_csv_path_, output_point, output_raw_point)
        ICESat_points_list.append(data.to_crs(epsg=4326))
    
    # Concatenate all points and prepare training data
    data = pd.concat(ICESat_points_list)
    y_train = data['OH'].values
    x_train = data[target_band].values
    
    # Train model
    if method == 'RF':
        best_params, classifier = optimize_random_forest(x_train, y_train)
        classifier.fit(x_train, y_train)
    elif method == 'Linear':
        classifier = np.polyfit(x_train[:, 0], y_train, deg=1)
    else:
        classifier = np.polyfit(x_train[:, 0], y_train, deg=2)
    
    # Apply model to each year's raster
    for year in year_list:
        FB_raster_path = f"G:/ICESat-2/{region}_map/{location}_map/A{year}/{location}_{year}_FB6_new9.tif"
        output_path = f'G:/ICESat-2/{region}_map/{location}_map/A{year}/{location}_{year}_{band}_{method}_FB6_new9_bys.tif'

        if not os.path.exists(FB_raster_path):
            print(FB_raster_path, 'does not exist!')
            break
        
        # Load raster and select target bands
        raster = rxr.open_rasterio(FB_raster_path).values
        raster = np.nan_to_num(raster, nan=0)
        selected_raster = raster[target_id, :, :]

        if len(selected_raster.shape) != 3:
            row, col = selected_raster.shape
            selected_raster = selected_raster.reshape(1, row, col)
        else:
            bands, row, col = selected_raster.shape

        print('selected_raster.shape', selected_raster.shape)

        # Predict elevation row by row
        k = 0
        result = np.array([[]])
        for i in range(row):
            x_input = np.transpose(selected_raster[:, i, :])
            if method == 'RF':
                y_pred = classifier.predict(x_input)
            else:
                y_pred = np.polyval(classifier, x_input)
            
            result = np.append(result, y_pred)
            
            if i % (int(row / 5)) == 0:
                print(k * 20, '% done')
                k += 1
        
        result = result.reshape(row, col)
        print(result.shape, type(result))
        
        # Mask tidal flats
        tf_mask = raster[6, :, :]
        result[(tf_mask == 0)] = 0

        # Align LULC with raster
        z = rxr.open_rasterio(FB_raster_path, masked=True).squeeze()
        LULC_aligned = LULC.rio.reproject_match(z)
        
        # Save result
        save_img(output_path, FB_raster_path, result)
        print(f'{location}_{year} output to {output_path}')